<a href="https://colab.research.google.com/github/VENOMDANGER2004/UCS-547-ACCELERATEDDS-/blob/main/ASSIGNDS4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Q1

In [1]:
import numpy as np
from numba import cuda
import time
import math

# CUDA Kernel
@cuda.jit
def compute_kernel(x, out):
    idx = cuda.grid(1)
    if idx < x.size:
        out[idx] = x[idx]**2 + 3*x[idx] + 5

# CPU Baseline Function
def compute_cpu(x):
    return x**2 + 3*x + 5

def run_q1():
    N = 5000000
    threadsperblock = 256
    blockspergrid = math.ceil(N / threadsperblock)

    for dtype in [np.float32, np.float64]:
        print(f"--- Testing with {dtype.__name__} ---")
        x_cpu = np.random.rand(N).astype(dtype)
        out_cpu = np.zeros_like(x_cpu)
        out_gpu = np.zeros_like(x_cpu)

        # CPU Execution
        start = time.time()
        out_cpu = compute_cpu(x_cpu)
        cpu_time = time.time() - start

        # GPU Execution
        d_x = cuda.to_device(x_cpu)
        d_out = cuda.to_device(out_gpu)

        # Warmup run to compile
        compute_kernel[blockspergrid, threadsperblock](d_x, d_out)

        start = time.time()
        compute_kernel[blockspergrid, threadsperblock](d_x, d_out)
        d_out.copy_to_host(out_gpu)
        gpu_time = time.time() - start

        print(f"CPU Time: {cpu_time:.4f}s")
        print(f"GPU Time: {gpu_time:.4f}s")
        print(f"Speedup: {cpu_time/gpu_time:.2f}x\n")

if __name__ == '__main__':
    run_q1()

--- Testing with float32 ---
CPU Time: 0.0267s
GPU Time: 0.0115s
Speedup: 2.32x

--- Testing with float64 ---
CPU Time: 0.0475s
GPU Time: 0.0098s
Speedup: 4.84x



Q2

In [2]:
import numpy as np
import numba as nb
import time

N = 1000000
data = np.random.rand(N)
bins = 100

def hist_python(data, bins):
    hist = [0] * bins
    for val in data:
        idx = int(val * bins)
        if idx == bins: idx -= 1
        hist[idx] += 1
    return hist

def hist_numpy(data, bins):
    return np.histogram(data, bins=bins, range=(0, 1))[0]

@nb.njit
def hist_numba(data, bins):
    hist = np.zeros(bins, dtype=np.int64)
    for i in range(data.size):
        idx = int(data[i] * bins)
        if idx == bins:
            idx -= 1
        hist[idx] += 1
    return hist

# Warmup Numba
_ = hist_numba(np.array([0.5]), bins)

start = time.time()
hist_python(data.tolist(), bins)
print(f"Pure Python: {time.time() - start:.4f}s")

start = time.time()
hist_numpy(data, bins)
print(f"NumPy: {time.time() - start:.4f}s")

start = time.time()
hist_numba(data, bins)
print(f"Numba: {time.time() - start:.4f}s")

Pure Python: 0.1858s
NumPy: 0.0161s
Numba: 0.0017s


Q3

In [3]:
import random
import numba as nb
import time

def monte_carlo_pi_py(nsamples):
    inside = 0
    for _ in range(nsamples):
        x = random.random()
        y = random.random()
        if x**2 + y**2 < 1.0:
            inside += 1
    return (inside / nsamples) * 4

@nb.njit
def monte_carlo_pi_nb(nsamples):
    inside = 0
    for _ in range(nsamples):
        x = random.random()
        y = random.random()
        if x**2 + y**2 < 1.0:
            inside += 1
    return (inside / nsamples) * 4

samples = 5000000

start = time.time()
pi_py = monte_carlo_pi_py(samples)
py_time = time.time() - start

# First run (includes compilation time)
start = time.time()
pi_nb = monte_carlo_pi_nb(samples)
nb_first_time = time.time() - start

# Second run (cached compilation)
start = time.time()
pi_nb = monte_carlo_pi_nb(samples)
nb_second_time = time.time() - start

print(f"Python Time: {py_time:.4f}s")
print(f"Numba First Time: {nb_first_time:.4f}s")
print(f"Numba Second Time: {nb_second_time:.4f}s")
print(f"Speedup Factor: {py_time / nb_second_time:.2f}x")

Python Time: 1.1865s
Numba First Time: 0.2450s
Numba Second Time: 0.0524s
Speedup Factor: 22.63x


Q4

In [4]:
import numpy as np
from numba import vectorize
import time

# a) Standard Vectorize
@vectorize
def adjust_brightness(pixel_value):
    new_val = pixel_value * 1.2
    return 255 if new_val > 255 else new_val

# c) Parallel Vectorize
@vectorize(['int64(int64)'], target='parallel')
def adjust_brightness_parallel(pixel_value):
    new_val = pixel_value * 1.2
    return 255 if new_val > 255 else int(new_val)

pixels = np.random.randint(0, 256, size=10000000, dtype=np.int64)

start = time.time()
_ = adjust_brightness(pixels)
print(f"Standard Vectorize: {time.time() - start:.4f}s")

start = time.time()
_ = adjust_brightness_parallel(pixels)
print(f"Parallel Vectorize: {time.time() - start:.4f}s")

Standard Vectorize: 0.0823s
Parallel Vectorize: 0.0294s


Q5

In [5]:
import numpy as np
import numba as nb
import time

# Generate synthetic data
N = 100000
features = 10
X = np.random.randn(N, features)
y = np.random.choice([-1, 1], size=N)
w_init = np.zeros(features)

# a) Standard NumPy
def log_reg_numpy(X, y, w, lr=0.01, iters=100):
    w = w.copy()
    for _ in range(iters):
        predictions = 1 / (1 + np.exp(-y * np.dot(X, w)))
        gradient = -np.dot(X.T, y * (1 - predictions)) / len(y)
        w -= lr * gradient
    return w

# b) Numba JIT Acceleration
@nb.njit
def log_reg_numba(X, y, w, lr=0.01, iters=100):
    w = w.copy()
    n_samples = X.shape[0]
    for _ in range(iters):
        gradient = np.zeros_like(w)
        for i in range(n_samples):
            margin = y[i] * np.dot(X[i], w)
            pred = 1.0 / (1.0 + np.exp(-margin))
            gradient -= y[i] * X[i] * (1.0 - pred)
        gradient /= n_samples
        w -= lr * gradient
    return w

# c) Compare correctness and performance
start = time.time()
w_np = log_reg_numpy(X, y, w_init)
np_time = time.time() - start

# Warmup
_ = log_reg_numba(X[:10], y[:10], w_init)

start = time.time()
w_nb = log_reg_numba(X, y, w_init)
nb_time = time.time() - start

print(f"NumPy Time: {np_time:.4f}s")
print(f"Numba Time: {nb_time:.4f}s")
print(f"Correctness (Weights Match): {np.allclose(w_np, w_nb)}")

NumPy Time: 0.3701s
Numba Time: 1.0370s
Correctness (Weights Match): True


Q6

In [6]:
import numpy as np
from numba import cuda
import math

@cuda.jit
def mat_add_kernel(A, B, C):
    row, col = cuda.grid(2)
    if row < C.shape[0] and col < C.shape[1]:
        C[row, col] = A[row, col] + B[row, col]

def run_q6():
    N = 1024
    A = np.random.rand(N, N).astype(np.float32)
    B = np.random.rand(N, N).astype(np.float32)
    C = np.zeros_like(A)

    # Configure grid and block dimensions
    threadsperblock = (16, 16)
    blockspergrid_x = math.ceil(A.shape[0] / threadsperblock[0])
    blockspergrid_y = math.ceil(A.shape[1] / threadsperblock[1])
    blockspergrid = (blockspergrid_x, blockspergrid_y)

    # Transfer to device
    d_A = cuda.to_device(A)
    d_B = cuda.to_device(B)
    d_C = cuda.to_device(C)

    # Run kernel
    mat_add_kernel[blockspergrid, threadsperblock](d_A, d_B, d_C)

    # Retrieve result
    d_C.copy_to_host(C)

    # Verification
    print("Matrix Addition successful:", np.allclose(A + B, C))

if __name__ == '__main__':
    run_q6()

Matrix Addition successful: True
